# Getting Started with DelaDect

This notebook walks through:

1. Building a specimen the easy way with `Specimen.from_cross_ply`
2. Running crack detection
3. Delamination detection using the `detector.edge` / `detector.diffuse` peer sub-detectors, plus the combined `detect_both_delaminations`
4. Saving and reloading a specimen
5. Building the **same** cross-ply specimen **manually** (plies + interface set up by hand) instead of using `Specimen.from_cross_ply`

Paths are resolved relative to the repo root automatically (see the first code cell), so this notebook works whether Jupyter's working directory is the repo root or this `notebooks/` folder itself.

## Path handling (hide me)

In [3]:
from pathlib import Path

from deladect.detection import crack_eval_crossply, DelaminationDetector
from deladect.specimen import Specimen
from skimage.io import imread


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    """Walk upward from the current working directory to find the repo root.

    Works regardless of whether Jupyter's working directory is the repo root
    or this notebook's own folder (``notebooks/``), which is what most
    Jupyter frontends default to.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root (missing {marker!r}) above {Path.cwd()}")


repo_root = find_repo_root()
data_root = repo_root / "example_images" / "sample-1" #/ "full"
frame_paths = sorted(data_root.glob("*.png"))
frame_paths

[WindowsPath('c:/Users/p2321038/Documents/GitHub/DelaDect/example_images/sample-1/0002_sc.png'),
 WindowsPath('c:/Users/p2321038/Documents/GitHub/DelaDect/example_images/sample-1/0092_sc.png'),
 WindowsPath('c:/Users/p2321038/Documents/GitHub/DelaDect/example_images/sample-1/0189_sc.png'),
 WindowsPath('c:/Users/p2321038/Documents/GitHub/DelaDect/example_images/sample-1/0284_sc.png'),
 WindowsPath('c:/Users/p2321038/Documents/GitHub/DelaDect/example_images/sample-1/0378_sc.png')]

## 1. Building your first specimen

`from_cross_ply` is a convenience function: it builds a `Specimen`, automatically adds two plies at `[0, 90]` degrees, and adds a single `"0/90"` interface between them.

In [4]:
specimen = Specimen.from_cross_ply(
    name="sample-1-quickstart",
    scale_px_mm=41.03328366,
    path_full=str(data_root),
    sorting_key="_sc",
    image_types=["png"],
    auto_init_stacks=True,
    results_root=str(repo_root / "results"),
    avg_crack_width_px=8.0,
)

[ply.name for ply in specimen.plies]

['ply_0', 'ply_1']

## 2. Run crack detection

`crack_eval_crossply` returns a dict keyed by orientation (`"0"`, `"90"`), each holding its own `cracks` / `densities` / `thresholds` / `metrics` / `paths` / `params`.

In [ ]:
crack_results = crack_eval_crossply(
    specimen,
    export_images=True,
    save_cracks=True,
    background = True,
    min_crack_size_px=60
)

crack_results.keys()

TypeError: crack_eval_crossply() got an unexpected keyword argument 'min_crack_length_px'

## 3. Delamination detection

`DelaminationDetector` exposes edge and diffuse detection as two peer sub-detectors:

- `detector.edge.detect_primary(...)` — edge-only detection
- `detector.diffuse.diffuse_delamination(...)` — diffuse-only detection (crack-guided ROIs)
- `detector.detect_both_delaminations(...)` — combined edge + diffuse, with overlap arbitration

Shared infrastructure (preprocessing, caching) lives directly on `DelaminationDetector` and is reused by both sub-detectors.

In [6]:
# from_cross_ply already added a single "0/90" interface between the two plies
interface = specimen.interfaces[0]
detector = DelaminationDetector(specimen, interface, save_preprocess_outputs=True)

# Use both cross-ply crack families as diffuse ROI input
cracks = Specimen.join_cracks(
    crack_results["0"]["cracks"],
    crack_results["90"]["cracks"],
)

cache_paths = detector.preprocess_stack_to_disk(
    specimen.image_stack_full,
    key="sample1_quickstart",
    reference_mode="rolling_median",
    reference_window=3,
    reference_skip=1,
)["cache_paths"]

len(cache_paths)

5

### 3a. Edge-only detection — `detector.edge.detect_primary`

In [7]:
edge_result = detector.edge.detect_primary(
    processed_cache_paths=cache_paths,
    save_overlays=False,
    debug=True,
)

edge_result.keys()

dict_keys(['masks', 'debug'])

### 3b. Diffuse-only detection — `detector.diffuse.diffuse_delamination`

In [8]:
diffuse_result = detector.diffuse.diffuse_delamination(
    cracks=cracks,
    processed_cache_paths=cache_paths,
    save_overlays=False,
    debug=True,
)

diffuse_result.keys()

dict_keys(['masks', 'debug'])

### 3c. Combined edge + diffuse — `detect_both_delaminations`

This is the orchestrator: it runs edge detection via `self.edge`, diffuse crack-tracking via `self.diffuse`, and resolves overlaps in favour of edge damage.

In [9]:
result = detector.detect_both_delaminations(
    cracks=cracks,
    avg_crack_width_px=8.0,
    processed_cache_paths=cache_paths,
    diffuse_params={
        "window_diffuse": (60, 60),
    },
    save_overlays=True,
    overlay_view="classified",
    save_masks=True,
    save_metrics=True,
    edge_exclusion_px=5,
    return_masks=False,
)

result["paths"]

ValueError: results directory part must be one non-empty relative path component; got 'edge_primary_auto_0/90'.

## 4. Save and reload the specimen

The specimen manifest records paths and configuration so a later session can reload cracks/delamination results without recomputation.

In [ ]:
from deladect.io import load_specimen, save_specimen

manifest = specimen.results_dir("config") / "sample-1-quickstart.json"
save_specimen(specimen, manifest)

specimen_reloaded = load_specimen(
    manifest,
    auto_init_stacks=False,
    load_results=True,
    verbose=True,
)

specimen_reloaded.name

## 5. Building the same cross-ply specimen manually

`Specimen.from_cross_ply` is convenience sugar over three calls: construct a plain `Specimen`, then call `add_ply` once per orientation. Below is the fully manual, equivalent construction — including adding the interface by hand — followed by the same crack + delamination workflow to confirm it behaves identically.

In [ ]:
specimen_manual = Specimen(
    name="sample-1-manual",
    scale_px_mm=41.03328366,
    path_full=str(data_root),
    sorting_key="_sc",
    image_types=["png"],
    auto_init_stacks=False,
    results_root=str(repo_root / "results"),
    avg_crack_width_px=8.0,
)

specimen_manual.path_full_list = [str(path) for path in frame_paths]
specimen_manual.image_stack_full = [imread(str(path)) for path in frame_paths]

# Manually add the two plies that from_cross_ply would have added automatically
specimen_manual.add_ply(name="ply_0", orientation_deg=0.0, avg_crack_width_px=8.0, min_crack_length_px=20.0)
specimen_manual.add_ply(name="ply_90", orientation_deg=90.0, avg_crack_width_px=8.0, min_crack_length_px=20.0)

# Manually add the interface between the two plies (index 0 = ply_0, index 1 = ply_90)
interface_manual = specimen_manual.add_interface(name="i0", upper_ply_index=0, lower_ply_index=1)

[ply.name for ply in specimen_manual.plies], interface_manual.name

In [ ]:
crack_results_manual = crack_eval_crossply(
    specimen_manual,
    export_images=True,
    save_cracks=True,
)

crack_results_manual.keys()

In [ ]:
detector_manual = DelaminationDetector(specimen_manual, interface_manual, save_preprocess_outputs=True)

cracks_manual = Specimen.join_cracks(
    crack_results_manual["0"]["cracks"],
    crack_results_manual["90"]["cracks"],
)

cache_paths_manual = detector_manual.preprocess_stack_to_disk(
    specimen_manual.image_stack_full,
    key="sample1_manual",
    reference_mode="rolling_median",
    reference_window=3,
    reference_skip=1,
)["cache_paths"]

result_manual = detector_manual.detect_both_delaminations(
    cracks=cracks_manual,
    avg_crack_width_px=8.0,
    processed_cache_paths=cache_paths_manual,
    diffuse_params={
        "window_diffuse": (60, 60),
    },
    save_overlays=True,
    overlay_view="classified",
    save_masks=True,
    save_metrics=True,
    edge_exclusion_px=5,
    return_masks=False,
)

result_manual["paths"]